# CatBoost


CatBoost is an advanced implementation of the gradient boosting framework designed to handle categorical data effectively while maintaining strong predictive performance.  
(Refer to [XGBoost](XGBoost.ipynb) for  gradient boosting.)


## Handling categorical features
A key challenge in machine learning is converting categorical variables into numerical form without losing useful information.

CatBoost addresses this using target-based encoding (also called target statistics).

### Target-based encoding

Instead of using simple encodings like one-hot encoding, CatBoost replaces each category with a value derived from the target.

Formally, for a category value $c$, the encoding can be written as:

$$
\text{Enc}(c) = \frac{\sum y_i + \alpha}{N_c + \alpha}
$$

- $N_c$ → number of samples with category $c$  
- $\alpha$ → smoothing parameter to avoid instability for small counts  

This allows the model to capture how strongly a category is associated with the target.

But if we compute this statistic using the entire dataset, the model indirectly uses the true target values of the same data point during training.  
This leads to data leakage, where the model gets unrealistic information and overfits.

### Ordered encoding (CatBoost solution)

To prevent this, CatBoost uses ordered statistics, where the dataset is randomly permuted and for each data point, the encoding is computed using only the samples that come before it,  
ensuring that the current sample’s target is not used in its own encoding.

This ensures that no future information is used, the encoding remains unbiased, and overfitting is reduced.

### Ordered boosting

In standard gradient boosting, the same dataset is used to compute residuals and fit the next tree, which can introduce bias and lead to overfitting.

This algorithm addresses this using ordered boosting.

In this approach, multiple random permutations of the dataset are created, and for each step, predictions are computed using only earlier samples in the permutation.  
The residuals are then calculated without using future information.

This ensures that each prediction is made in a way that simulates real-world inference, reducing overfitting and improving generalization.

### Tree structure

The algorithm uses symmetric trees where the same split condition is applied at each level, and all nodes at the same depth use the same feature and threshold.

This results in a balanced tree structure.  
It also provides faster training due to the simplified structure, faster prediction due to fixed decision paths, and built-in regularization since the model avoids overly complex trees.

## Objective function

CatBoost minimizes a regularized objective similar to other boosting methods:

$$
\text{Objective} = \sum_{i=1}^{n} l(y_i, \hat{y}_i) + \sum_{k=1}^{K} \Omega(f_k)
$$

- $l(y_i, \hat{y}_i)$ → loss function (e.g., MSE for regression, log loss for classification)  
- $\Omega(f_k)$ → regularization term controlling tree complexity  

This balances fitting the data well while preventing overfitting.


## Training process

The training process proceeds iteratively:

1. initialize predictions (usually a constant value)  
2. compute residuals based on current predictions  
3. compute ordered target statistics for categorical features  
4. build trees using symmetric structure  
5. update predictions by adding the new tree  
6. repeat until convergence or stopping criteria  






In [19]:
import kagglehub
import pandas as pd
path = kagglehub.dataset_download("akshatgupta7/crop-yield-in-indian-states-dataset")
dataset = pd.read_csv(path + "/crop_yield.csv")
dataset.head()

,Crop,Crop_Year,Season,State,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Yield
0,Arecanut,1997,Whole Year,Assam,73814.0,56708,2051.4,7024878.38,22882.34,0.796087
1,Arhar/Tur,1997,Kharif,Assam,6637.0,4685,2051.4,631643.29,2057.47,0.710435
2,Castor seed,1997,Kharif,Assam,796.0,22,2051.4,75755.32,246.76,0.238333
3,Coconut,1997,Whole Year,Assam,19656.0,126905000,2051.4,1870661.52,6093.36,5238.051739
4,Cotton(lint),1997,Kharif,Assam,1739.0,794,2051.4,165500.63,539.09,0.420909


In [18]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

X = dataset.drop("Yield", axis=1)
y = dataset["Yield"]


cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

catboost_model = CatBoostRegressor(iterations=1000, learning_rate=0.1, depth=6, random_state=42)

catboost_model.fit(X_train, y_train, cat_features=cat_features, verbose=100)

y_pred_catboost = catboost_model.predict(X_test)

mse_catboost = mean_squared_error(y_test, y_pred_catboost)
r2_score_catboost = r2_score(y_test, y_pred_catboost)

print("Regression Mean Squared Error:", mse_catboost)
print("Regression R^2 Score:", r2_score_catboost)

0:	learn: 804.5047271	total: 79.1ms	remaining: 1m 18s
100:	learn: 77.2206748	total: 4.66s	remaining: 41.5s
200:	learn: 46.7042615	total: 9.17s	remaining: 36.5s
300:	learn: 35.0711244	total: 13.6s	remaining: 31.5s
400:	learn: 26.5623255	total: 18s	remaining: 26.9s
500:	learn: 22.2786274	total: 22.5s	remaining: 22.4s
600:	learn: 19.0119434	total: 27.1s	remaining: 18s
700:	learn: 16.0822920	total: 31.7s	remaining: 13.5s
800:	learn: 13.3423633	total: 36.3s	remaining: 9.02s
900:	learn: 11.8326269	total: 40.8s	remaining: 4.48s
999:	learn: 10.6500258	total: 45.1s	remaining: 0us
Regression Mean Squared Error: 24168.34176324037
Regression R^2 Score: 0.9698363214041902


In [23]:
#CLASSIFICATION
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.datasets import load_wine
wine_data = load_wine()
X = wine_data.data
y = wine_data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
catboost_classifier = CatBoostClassifier(iterations=1000, learning_rate=0.1, depth=6, random_state=42)
catboost_classifier.fit(X_train, y_train, verbose=100)
y_pred_catboost = catboost_classifier.predict(X_test)
accuracy_catboost = accuracy_score(y_test, y_pred_catboost)
print("Classification Accuracy:", accuracy_catboost)
print("Classification Report:\n", classification_report(y_test, y_pred_catboost))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_catboost))

0:	learn: 1.0125552	total: 2.56ms	remaining: 2.55s
100:	learn: 0.0452673	total: 159ms	remaining: 1.41s
200:	learn: 0.0171715	total: 299ms	remaining: 1.19s
300:	learn: 0.0105663	total: 423ms	remaining: 981ms
400:	learn: 0.0075918	total: 546ms	remaining: 815ms
500:	learn: 0.0059501	total: 674ms	remaining: 672ms
600:	learn: 0.0048346	total: 801ms	remaining: 532ms
700:	learn: 0.0041353	total: 939ms	remaining: 400ms
800:	learn: 0.0035893	total: 1.07s	remaining: 267ms
900:	learn: 0.0031561	total: 1.2s	remaining: 132ms
999:	learn: 0.0028263	total: 1.33s	remaining: 0us
Classification Accuracy: 1.0
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        14
           1       1.00      1.00      1.00        14
           2       1.00      1.00      1.00         8

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36


## Key Parameters

**iterations**

  Number of trees to build.

**learning_rate**

  Same role as  the boosting frameworks param.


**depth**

  Tree depth.


**l2_leaf_reg**

  L2 regularization on leaf values.

  - larger → smoother predictions  
  - smaller → more flexible model  

**loss_function**

  regression → `'RMSE'`  
  classification → `'Logloss'`, `'MultiClass'`  


**border_count**

  number of splits for numerical features  


**cat_features**  

  Specifies categorical feature indices.


## Docs

https://catboost.ai/docs/en/concepts/python-reference_catboostregressor

https://catboost.ai/docs/en/concepts/python-reference_catboostclassifier